<!-- NB_DOC_INTRO_V1 -->
# INRIA scikit-learn MOOC — notes condensees

> 📚 **Doc thematique** : [docs/03_ML.md](docs/03_ML.md) (Machine Learning classique)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

Notes du **MOOC INRIA scikit-learn** ([scikit-learn-mooc](https://inria.github.io/scikit-learn-mooc/)) — la reference de bonnes pratiques sklearn par l'equipe core. Ce notebook condense les points cles : **cross-validation**, **modeles ensemblistes** (Bagging, Random Forest, Gradient Boosting, HistGradientBoosting), **feature importance** (RFECV + permutation), **encoding** ordinal vs OneHot, **pipelines** anti-leakage.

Dataset : California Housing (regression) + UCI Adult Income (classification) — auto-DL.

## Plan

1. Setup + dataset
2. Cross-validation (KFold, StratifiedKFold, TimeSeriesSplit)
3. Random Forest (regression)
4. Gradient Boosting (sklearn vs HistGradientBoosting)
5. RFECV — Recursive Feature Elimination
6. Permutation importance (vs feature_importances_ naif)
7. Pipeline + ColumnTransformer (anti-leakage)
8. Ordinal vs OneHot encoding
9. Pieges courants
10. References


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import (
    cross_val_score, cross_validate, KFold, StratifiedKFold, TimeSeriesSplit,
    train_test_split,
)
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFECV
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, root_mean_squared_error, accuracy_score
import warnings
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

# California housing (regression)
ca = fetch_california_housing(as_frame=True)
X, y = ca.data, ca.target
print(f"California Housing : {X.shape}, features : {list(X.columns)}")

## 1. Cross-validation — les types

- **KFold** : split aleatoire (default)
- **StratifiedKFold** : preserve la distribution des classes (classification)
- **TimeSeriesSplit** : preserve l'ordre temporel (TS)
- **GroupKFold** : evite la fuite intra-groupe (ex: meme patient sur train + test)

In [ ]:
rf = RandomForestRegressor(n_estimators=50, random_state=SEED, n_jobs=-1)

# KFold 5 splits
kfold = KFold(n_splits=5, shuffle=True, random_state=SEED)
scores = cross_val_score(rf, X, y, cv=kfold, scoring="r2", n_jobs=-1)
print(f"KFold R² : {scores.mean():.4f} ± {scores.std():.4f}")

# cross_validate : multiples metriques + train/test scores
results = cross_validate(rf, X, y, cv=kfold,
                          scoring=["r2", "neg_root_mean_squared_error"],
                          return_train_score=True, n_jobs=-1)
print(f"\nTrain R²  : {results['train_r2'].mean():.4f}  (sur-evalue : overfit)")
print(f"Test  R²  : {results['test_r2'].mean():.4f}")
print(f"Test RMSE : {-results['test_neg_root_mean_squared_error'].mean():.4f}")

## 2. Random Forest

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=SEED)

rf = RandomForestRegressor(n_estimators=200, max_depth=20, random_state=SEED, n_jobs=-1)
rf.fit(X_tr, y_tr)

r2 = rf.score(X_te, y_te)
rmse = root_mean_squared_error(y_te, rf.predict(X_te))
print(f"RF — R² : {r2:.4f}, RMSE : {rmse:.4f}")

# Feature importances (impurity-based — biaise vers features a haute cardinalite, attention)
imps = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(f"\nFeature importances (impurity) :\n{imps}")

## 3. Gradient Boosting — sklearn vs HistGradientBoosting

**HistGradientBoosting** est l'equivalent sklearn de LightGBM : ultra-rapide (binning) + supporte les NaN nativement.

In [ ]:
# GBR classique : lent sur gros datasets
gbr = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=SEED)
gbr.fit(X_tr, y_tr)
print(f"GBR  — R² test : {gbr.score(X_te, y_te):.4f}")

# HistGBR : 10-100x plus rapide, NaN-native
hgbr = HistGradientBoostingRegressor(max_iter=100, max_depth=5, random_state=SEED)
hgbr.fit(X_tr, y_tr)
print(f"HGBR — R² test : {hgbr.score(X_te, y_te):.4f}")

# Demo NaN-native : on injecte des NaN dans X_tr
X_tr_nan = X_tr.copy()
X_tr_nan.iloc[:100, 0] = np.nan
hgbr_nan = HistGradientBoostingRegressor(max_iter=50, random_state=SEED)
hgbr_nan.fit(X_tr_nan, y_tr)
print(f"\nHGBR avec NaN dans train : R² test = {hgbr_nan.score(X_te, y_te):.4f}")

## 4. RFECV — Recursive Feature Elimination avec CV

Elimine recursivement les features les moins importantes jusqu'a trouver le **nombre optimal**.

In [ ]:
# Sur 5 splits CV, evalue combien de features sont vraiment utiles
selector = RFECV(
    estimator=RandomForestRegressor(n_estimators=50, random_state=SEED, n_jobs=-1),
    step=1,                    # supprime 1 feature par iteration
    min_features_to_select=2,
    cv=3,                       # 3 CV folds pour la rapidite
    scoring="r2",
    n_jobs=-1,
)
selector.fit(X_tr, y_tr)

print(f"Nb optimal features : {selector.n_features_}")
print(f"Features retenues : {X_tr.columns[selector.support_].tolist()}")
print(f"Score avec selection : {selector.score(X_te, y_te):.4f}")

## 5. Permutation importance — alternative robuste a `feature_importances_`

**Pourquoi** : `feature_importances_` (impurity-based) est **biaise** envers features a haute cardinalite. La permutation importance est model-agnostic et plus fiable.

In [ ]:
# Importance par permutation : pour chaque feature, on melange ses valeurs et on mesure
# la chute de performance. Une feature importante → chute grande.
result = permutation_importance(rf, X_te, y_te, n_repeats=10, random_state=SEED, n_jobs=-1)

imp_perm = pd.DataFrame({
    "feature": X.columns,
    "mean_drop_r2": result.importances_mean,
    "std": result.importances_std,
}).sort_values("mean_drop_r2", ascending=False)
print(imp_perm)

# Comparer impurity vs permutation
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(X.columns))
w = 0.35
ax.barh(x - w/2, imps.reindex(X.columns).values, w, label="Impurity")
ax.barh(x + w/2, imp_perm.set_index("feature")["mean_drop_r2"].reindex(X.columns).values, w, label="Permutation")
ax.set_yticks(x); ax.set_yticklabels(X.columns)
ax.legend(); ax.set_title("Feature importance : Impurity vs Permutation")
plt.tight_layout(); plt.show()

## 6. Encoding ordinal vs OneHot — quand utiliser quoi

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# UCI Adult Income (classification : revenue > 50k$ ?)
# On simule depuis California en ajoutant une feature cat
X_demo = X.copy()
X_demo["zone"] = pd.cut(X_demo["Latitude"], bins=3, labels=["sud", "centre", "nord"])

# OneHot (recommande pour LogReg, SVM, NN)
preproc_ohe = ColumnTransformer([
    ("ohe", OneHotEncoder(handle_unknown="ignore", drop="first"), ["zone"]),
], remainder="passthrough")

# Ordinal (recommande pour arbres — Random Forest, GBM)
preproc_ord = ColumnTransformer([
    ("ord", OrdinalEncoder(), ["zone"]),
], remainder="passthrough")

# Bench rapide
for name, preproc in [("OneHot + LogReg", preproc_ohe), ("Ordinal + RF", preproc_ord)]:
    model = LogisticRegression(max_iter=1000) if "LogReg" in name else RandomForestRegressor(n_estimators=50, random_state=SEED)
    pipe = Pipeline([("preproc", preproc), ("model", model)])
    if "LogReg" in name:
        # Binariser y pour LogReg
        y_bin = (y > y.median()).astype(int)
        score = cross_val_score(pipe, X_demo, y_bin, cv=3, scoring="accuracy", n_jobs=-1).mean()
        print(f"{name} : accuracy = {score:.4f}")
    else:
        score = cross_val_score(pipe, X_demo, y, cv=3, scoring="r2", n_jobs=-1).mean()
        print(f"{name} : R² = {score:.4f}")

## 7. Pieges courants (selon le MOOC INRIA)

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| **Scaler/Imputer avant le split** | Toujours en Pipeline (anti-leakage) |
| `cross_val_score(model, X, y)` sans pipeline | Si preprocessing requis : ColumnTransformer dans Pipeline |
| **Ordinal sur LogReg/SVM** | OneHotEncoder (sauf si vraiment ordinal et tu sais l'ordre) |
| `feature_importances_` comme seule reference | Croiser avec permutation_importance |
| `KFold` sur classification desequilibree | `StratifiedKFold` |
| `KFold` sur time series | `TimeSeriesSplit` (jamais melange du futur dans le train) |
| Garder toutes les features sans test | RFECV ou eliminer apres SHAP/permutation |
| `n_jobs=-1` partout sans tester | Parfois RAM explose (RF avec n_estimators=500, max_depth=None) |


## References

### Documentation
- INRIA scikit-learn MOOC : https://inria.github.io/scikit-learn-mooc/
- sklearn user guide : https://scikit-learn.org/stable/user_guide.html
- Permutation importance : https://scikit-learn.org/stable/modules/permutation_importance.html

### Voir aussi
- [ML_Regression_Classification_CV_GridSearch.ipynb](ML_Regression_Classification_CV_GridSearch.ipynb)
- [ML_Bagging_Boosting.ipynb](ML_Bagging_Boosting.ipynb)
- [ML_Explication_Feature_Importance_Selection.ipynb](ML_Explication_Feature_Importance_Selection.ipynb)
- [DS_Metrics_Reference.ipynb](DS_Metrics_Reference.ipynb)
- [Structures_Preprocessing.ipynb](Structures_Preprocessing.ipynb)
